1\. Write a function that converts number representation, bin<->dec<->hex. (Clearly using the corresponding python built-in functions is not fair..)

In [44]:
digits="0123456789ABCDEF"
def to_decimal(s, base): #CONVERT ANYTHING TO DECIMAL NUMBER, AFTERTHAT, WITH ANOTHER FUNC WE CONVERT IT TO HEX OR BIN.
    s = s.upper()
    value = 0
    for k in s:
        value = base * value + digits.index(k)
    return value

In [45]:
def from_decimal(n, base):
    if n == 0:
        return "0"
    DIGIT=[]
    while n>0:
        DIGIT.append(digits[n%base])
        n //= base
    return "".join(reversed(DIGIT))

In [46]:
def convert(s,from_base, to_base):
    dec = to_decimal(s,from_base)
    return (from_decimal(dec,to_base))

In [48]:
convert('FF',16,10)

'255'

In [3]:
int('FF',16)

255

2\. Write a function that converts a 32 bit word into a single precision floating point (i.e. interprets the various bits as sign, mantissa and exponent)

In [4]:
# https://chatgpt.com/share/693dd12f-daf8-8011-9e74-6d378ee309c5

In [51]:
word32_to_float(0x3F800000)

1.0

In [52]:
import math

def word32_to_float(w):
    w = w & 0xFF_FFF_FFF  # keep only 32 bits ("8 hex digits" * 4 bits = 32)

    # EXTRACT SECTIONS -> SIGN / EXPONENT / MANTISSA
    s = (w >> 31) & 0x1
    e = (w >> 23) & 0xFF
    f = w & 0x7FFFFF

    sign = -1.0 if s == 1 else 1.0
    frac = f / (1 << 23)  # m / 2^23

    # CASE 1: e == 255 -> infinity or NaN
    if e == 0xFF:
        if f == 0:
            return sign * math.inf
        return math.nan

    # CASE 2: e == 0 -> zero or subnormal
    if e == 0:
        if f == 0:
            return -0.0 if s == 1 else 0.0  # signed zero
        return sign * frac * (2.0 ** -126)   # subnormal (exponent = -126)

    # CASE 3: normal (1 <= e <= 254)
    mantisa = 1.0 + frac
    exponent = e - 127
    return sign * mantisa * (2.0 ** exponent)


In [50]:
word32_to_float(0x80000001)

-1.401298464324817e-45

3\. Write a program to determine the underflow and overflow limits (within a factor of 2) for python on your computer. 

**Tips**: define two variables inizialized to 1 and halve/double them enough time to exceed the under/over-flow limits  

In [9]:
import sys
print(sys.float_info)

sys.float_info(max=1.7976931348623157e+308, max_exp=1024, max_10_exp=308, min=2.2250738585072014e-308, min_exp=-1021, min_10_exp=-307, dig=15, mant_dig=53, epsilon=2.220446049250313e-16, radix=2, rounds=1)


In [10]:
x = 1.0
y = 1.0
while x > 0.0:
    previous = x
    x /= 2.0
print("underflow is occured",previous,"and", x)
while y != math.inf:
    prev = y
    y *= 2.0
print("overflow is occured",prev,"and", y)

underflow is occured 5e-324 and 0.0
overflow is occured 8.98846567431158e+307 and inf


4\. Write a program to determine the machine precision

**Tips**: define a new variable by adding a smaller and smaller value (proceeding similarly to prob. 2) to an original variable and check the point where the two are the same 

In [11]:
import sys
print(sys.float_info.epsilon)

eps = 1.0
while eps + 1.0 != 1.0:
    pre = eps
    eps /= 2.0
print("measurement the blindness of my pc", pre)

2.220446049250313e-16
measurement the blindness of my pc 2.220446049250313e-16


5\. Write a function that takes in input three parameters $a$, $b$ and $c$ and prints out the two solutions to the quadratic equation $ax^2+bx+c=0$ using the standard formula:
$$
x=\frac{-b\pm\sqrt{b^2-4ac}}{2a}
$$

(a) use the program to compute the solution for $a=0.001$, $b=1000$ and $c=0.001$

(b) re-express the standard solution formula by multiplying top and bottom by $-b\mp\sqrt{b^2-4ac}$ and again find the solution for $a=0.001$, $b=1000$ and $c=0.001$. How does it compare with what previously obtained? Why?

(c) write a function that compute the roots of a quadratic equation accurately in all cases

In [12]:
#(a) 
def f(a,b,c):
    delta = math.sqrt(b**2 - 4 * a * c)
    return (-b + delta) / (2 * a) ,  (-b - delta) / (2 * a)

In [13]:
#(b)
def m(a,b,c):
    delta = math.sqrt(b**2 - 4 * a * c)
    return (-b + delta) * (-b - delta) / (2 * a) * (-b - delta) , (-b - delta) * (-b + delta) / (2 * a) * (-b + delta)

In [53]:
#(c)
def r(a,b,c):
    delta = math.sqrt(b**2 - 4 * a * c)
    if b >= 0 :
        x1 = (-b - delta)/ (2 * a)
    else :
        x1 = (-b + delta)/ (2 * a)
    x2 = c/(a*x1)
    return x1,x2

In [54]:
print(f(0.001,1000,0.001), m(0.001,1000,0.001), r(0.001,1000,0.001))

(-9.999894245993346e-07, -999999.999999) (-3.999957698389339, -3.999915397238034e-12) (-999999.999999, -1.000000000001e-06)


6\. Write a program that implements the function $f(x)=x(x−1)$

(a) Calculate the derivative of the function at the point $x = 1$ using the derivative definition:

$$
\frac{{\rm d}f}{{\rm d}x} = \lim_{\delta\to0} \frac{f(x+\delta)-f(x)}{\delta}
$$

with $\delta = 10^{−2}$. Calculate the true value of the same derivative analytically and compare with the answer your program gives. The two will not agree perfectly. Why not?

(b) Repeat the calculation for $\delta = 10^{−4}, 10^{−6}, 10^{−8}, 10^{−10}, 10^{−12}$ and $10^{−14}$. How does the accuracy scales with $\delta$?

In [16]:
def d(x):
    return x * (x-1)
x = 1.0
delta = 1e-2
num = (d(x+delta) - d(x)) / delta
dev = 2*x-1
print("Numerical derivative",num)
print("Analytical derivative",dev)

Numerical derivative 1.010000000000001
Analytical derivative 1.0


7\. Consider the integral of the semicircle of radius 1:
$$
I=\int_{-1}^{1} \sqrt(1-x^2) {\rm d}x
$$
which it's known to be $I=\frac{\pi}{2}=1.57079632679...$.
Alternatively we can use the Riemann definition of the integral:
$$
I=\lim_{N\to\infty} \sum_{k=1}^{N} h y_k 
$$

with $h=2/N$ the width of each of the $N$ slices the domain is divided into and where
$y_k$ is the value of the function at the $k-$th slice.

(a) Write a programe to compute the integral with $N=100$. How does the result compares to the true value?

(b) How much can $N$ be increased if the computation needs to be run in less than a second? What is the gain in running it for 1 minute? 


In [17]:
n = 100
h = 2/n # 1 - (-1) / number of slices
s = 0.0
for k in range(n):
    x = -1 + (k + 0.5) * h # mid_point of each slice 
    s += math.sqrt(1-x**2)
I = h*s
r = math.pi / 2
print("Rieman estimate:", I, "real value",r, "error", I-r) 

Rieman estimate: 1.5712827762297954 real value 1.5707963267948966 error 0.00048644943489883907
